In [0]:
# 00_apply_migrations — sql/databricks/migrations/*.sql 을 순서대로 실행
# DDL 원본: sql/databricks/migrations/
# 이 노트북은 스키마/테이블 생성만 담당한다. 데이터 적재는 하지 않는다.

In [0]:
dbutils.widgets.text("catalog", "access_drift")

CATALOG = dbutils.widgets.get("catalog")
MIGRATIONS_BASE = "/Workspace/Shared/access-drift/files/sql/databricks/migrations"

In [0]:
import os

if not os.path.isdir(MIGRATIONS_BASE):
    raise RuntimeError(
        f"Migration 파일 디렉터리를 찾을 수 없습니다: {MIGRATIONS_BASE}\n"
        "bundle deploy가 완료됐는지 확인하세요."
    )

In [0]:
# migration 파일별 변수 치환 정의
def strip_line_comments(sql: str) -> str:
    return "\n".join(
        line for line in sql.splitlines()
        if not line.strip().startswith("--")
    )

migrations = [
    ("001_create_bronze_tables.sql", {"catalog": CATALOG, "schema": "bronze"}),
    ("002_create_silver_tables.sql", {"catalog": CATALOG, "schema": "silver"}),
    ("003_create_eval_tables.sql",   {"catalog": CATALOG, "schema": "eval"}),
    ("004_create_intermediate_tables.sql", {"catalog": CATALOG, "schema": "intermediate"}),
    ("005_create_gold_tables.sql",         {"catalog": CATALOG, "schema": "gold"}),
    ("006_create_context_tables.sql",      {"catalog": CATALOG, "schema": "context"}),
    ("007_create_app_tables.sql",          {"catalog": CATALOG, "schema": "app"}),
]

for filename, variables in migrations:
    path = os.path.join(MIGRATIONS_BASE, filename)
    with open(path) as f:
        sql = f.read()

    for key, value in variables.items():
        sql = sql.replace(f"${{{key}}}", value)

    sql = strip_line_comments(sql)
    statements = [
        s.strip() for s in sql.split(";")
        if s.strip()
    ]

    for stmt in statements:
        spark.sql(stmt)

    print(f"{filename}: {len(statements)} statements applied")

001_create_bronze_tables.sql: 7 statements applied
002_create_silver_tables.sql: 5 statements applied
003_create_eval_tables.sql: 2 statements applied
004_create_intermediate_tables.sql: 1 statements applied
005_create_gold_tables.sql: 1 statements applied


## Smoke test

In [0]:
expected_tables = {
    "bronze": [
        "bronze_azure_applications",
        "bronze_azure_application_owners",
        "bronze_azure_password_credentials",
        "bronze_identity_lifecycle_seed",
        "bronze_assets_seed",
        "bronze_access_edges_seed",
    ],
    "silver": [
        "silver_principals",
        "silver_credentials",
        "silver_assets",
        "silver_edges",
    ],
    "eval": [
        "ground_truth_case",
    ],
    "intermediate": [
        "nhi_residual_access_findings",
    ],
    "gold": [
        "gold_core",
    ],
    "context": [
        "asset_context",
        "assignment_routing",
    ],
    "app": [
        "risk_action_state",
    ],
}

for schema, tables in expected_tables.items():
    existing = {
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()
    }
    missing = sorted(set(tables) - existing)
    assert not missing, f"Missing tables in {CATALOG}.{schema}: {missing}"

print("migration smoke test passed")
print("migrations complete")

migration smoke test passed
migrations complete
